# PlaceMux - Matching Demo
Week 2 task - building the job matching foundation

This notebook walks through the matching logic, shows a single student-job example, and evaluates performance on the full dataset.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from matching import (
    calculate_match,
    top_jobs_for_student,
    top_students_for_job,
    full_match_matrix,
    evaluate_matching,
    feature_space_summary,
    SKILL_FEATURES,
    SKILL_THRESHOLD
)

## Load data

In [2]:
students_df = pd.read_csv('../data/students.csv')
jobs_df = pd.read_csv('../data/jobs.csv')

print(f'students: {len(students_df)}')
print(f'jobs: {len(jobs_df)}')
print(f'total pairs to evaluate: {len(students_df) * len(jobs_df)}')

students: 30
jobs: 20
total pairs to evaluate: 600


In [3]:
students_df.head()

,student_id,name,python,sql,machine_learning,communication,data_visualization,cloud,projects,internships,cgpa
0,1,Aarav Shah,90,85,80,75,70,60,4,1,8.5
1,2,Priya Patel,70,60,50,80,65,40,2,0,7.2
2,3,Rohan Mehta,95,92,88,90,85,78,6,2,9.1
3,4,Sneha Iyer,55,70,40,85,50,30,1,0,6.8
4,5,Karan Joshi,80,75,72,65,80,55,3,1,8.0


In [4]:
jobs_df.head()

,job_id,company,title,python,sql,machine_learning,communication,data_visualization,cloud,min_projects,min_internships,min_cgpa,salary_lpa,location
0,101,TechCorp India,Data Scientist,1,1,1,1,1,0,3,1,8.0,12,Bangalore
1,102,FinSoft Solutions,Data Analyst,1,1,0,1,1,0,2,0,7.0,8,Mumbai
2,103,CloudBase Pvt Ltd,ML Engineer,1,0,1,0,1,1,4,1,8.5,15,Hyderabad
3,104,StartupXYZ,Python Developer,1,0,0,1,0,0,2,0,7.5,7,Pune
4,105,HealthTech Ltd,Healthcare Data Analyst,0,1,0,1,1,0,2,1,7.5,9,Chennai


## Feature space

In [5]:
# what features are we using and how
feature_space_summary()

,Feature,Type,Student Field,Job Field,Match Rule,Weight
0,Python,Skill Score,python (0-100),python (0 or 1),student.python >= 70,25%
1,Sql,Skill Score,sql (0-100),sql (0 or 1),student.sql >= 70,20%
2,Machine Learning,Skill Score,machine_learning (0-100),machine_learning (0 or 1),student.machine_learning >= 70,25%
3,Communication,Skill Score,communication (0-100),communication (0 or 1),student.communication >= 70,10%
4,Data Visualization,Skill Score,data_visualization (0-100),data_visualization (0 or 1),student.data_visualization >= 70,10%
5,Cloud,Skill Score,cloud (0-100),cloud (0 or 1),student.cloud >= 70,10%
6,Projects,Profile,projects (count),min_projects (count),student.projects >= job.min_projects,part of 20% profile
7,Internships,Profile,internships (count),min_internships (count),student.internships >= job.min_internships,part of 20% profile
8,CGPA,Profile,cgpa (0-10),min_cgpa (0-10),student.cgpa >= job.min_cgpa,part of 20% profile


## Quick EDA

In [6]:
# skill score distribution across students
students_df[SKILL_FEATURES].describe().round(1)

,python,sql,machine_learning,communication,data_visualization,cloud
count,30.0,30.0,30.0,30.0,30.0,30.0
mean,77.1,75.4,68.6,77.9,71.0,53.5
std,13.4,11.7,16.2,8.1,13.0,16.0
min,50.0,55.0,35.0,65.0,45.0,25.0
25%,65.8,65.8,55.8,70.5,60.5,40.5
50%,79.0,77.0,71.0,79.0,72.0,53.5
75%,88.0,85.0,81.5,85.0,81.5,63.5
max,98.0,95.0,92.0,90.0,90.0,85.0


In [7]:
# what % of students clear the 70 threshold per skill
print(f'% students above threshold ({SKILL_THRESHOLD}) per skill:')
for skill in SKILL_FEATURES:
    pct = (students_df[skill] >= SKILL_THRESHOLD).mean() * 100
    bar = '#' * int(pct // 5)
    print(f'  {skill:<25} {bar:<20} {pct:.0f}%')

% students above threshold (70) per skill:
  python                    ##############       70%
  sql                       ##############       70%
  machine_learning          ###########          57%
  communication             ################     83%
  data_visualization        ############         60%
  cloud                     ####                 20%


In [8]:
# how often does each skill appear as a job requirement
print('% of jobs requiring each skill:')
for skill in SKILL_FEATURES:
    pct = jobs_df[skill].mean() * 100
    bar = '#' * int(pct // 5)
    print(f'  {skill:<25} {bar:<20} {pct:.0f}%')

% of jobs requiring each skill:
  python                    ##############       70%
  sql                       ###############      75%
  machine_learning          ##########           50%
  communication             ###############      75%
  data_visualization        ###############      75%
  cloud                     ######               30%


## Single pair demo
This is what the founder will want to see - one student, one job, explain the result.

In [9]:
student = students_df[students_df['student_id'] == 1].iloc[0]
job = jobs_df[jobs_df['job_id'] == 101].iloc[0]

result = calculate_match(student, job)

print('='*55)
print(f'Student : {student["name"]} (ID {result.student_id})')
print(f'Job     : {job["title"]} @ {job["company"]} (ID {result.job_id})')
print(f'Location: {job["location"]}  |  Salary: Rs.{job["salary_lpa"]} LPA')
print('='*55)
print(f'Match Score  : {result.match_score}%')
print(f'Skill Score  : {result.skill_score}%')
print(f'Profile Score: {result.profile_score}%')
print(f'Status       : {result.status}')
print()
print('Breakdown:')
for r in result.reason:
    print(f'  {r}')
if result.warnings:
    print('Warnings:')
    for w in result.warnings:
        print(f'  ! {w}')

Student : Aarav Shah (ID 1)
Job     : Data Scientist @ TechCorp India (ID 101)
Location: Bangalore  |  Salary: Rs.12 LPA
Match Score  : 100.0%
Skill Score  : 100.0%
Profile Score: 100.0%
Status       : Highly Recommended

Breakdown:
  Python: 90/100 ✓
  Sql: 85/100 ✓
  Machine Learning: 80/100 ✓
  Communication: 75/100 ✓
  Data Visualization: 70/100 ✓
  Projects: 4 (needed >= 3) ✓
  Internships: 1 (needed >= 1) ✓
  CGPA: 8.5 (needed >= 8.0) ✓


## Partial match example - student missing some skills

In [10]:
# student 2 has weaker scores, let's see what happens against same job
student2 = students_df[students_df['student_id'] == 2].iloc[0]
result2 = calculate_match(student2, job)

print(f'Student : {student2["name"]}')
print(f'Score   : {result2.match_score}%  ->  {result2.status}')
print(f'Matched : {result2.matched_skills}')
print(f'Missing : {result2.missing_skills}')
print()
for r in result2.reason:
    print(f'  {r}')
if result2.warnings:
    for w in result2.warnings:
        print(f'  ! {w}')

Student : Priya Patel
Score   : 31.11%  ->  Not Recommended
Matched : ['python', 'communication']
Missing : ['sql', 'machine_learning', 'data_visualization']

  Python: 70/100 ✓
  Sql: 60/100 (below 70) ✗
  Machine Learning: 50/100 (below 70) ✗
  Communication: 80/100 ✓
  Data Visualization: 65/100 (below 70) ✗
  ! Projects: 2 (needed >= 3) ✗
  ! Internships: 0 (needed >= 1) ✗
  ! CGPA: 7.2 (needed >= 8.0) ✗


## Top 5 jobs for a student

In [11]:
student = students_df[students_df['student_id'] == 1].iloc[0]
top_jobs = top_jobs_for_student(student, jobs_df, top_n=5)

print(f'Top 5 jobs for {student["name"]}:\n')
print(f'{"#":<4} {"Job":<35} {"Company":<22} {"Score":<8} Status')
print('-'*85)
for i, r in enumerate(top_jobs, 1):
    j = jobs_df[jobs_df['job_id'] == r.job_id].iloc[0]
    print(f'{i:<4} {j["title"]:<35} {j["company"]:<22} {r.match_score:<8} {r.status}')

Top 5 jobs for Aarav Shah:

#    Job                                 Company                Score    Status
-------------------------------------------------------------------------------------
1    Data Scientist                      TechCorp India         100.0    Highly Recommended
2    Data Analyst                        FinSoft Solutions      100.0    Highly Recommended
3    Python Developer                    StartupXYZ             100.0    Highly Recommended
4    Healthcare Data Analyst             HealthTech Ltd         100.0    Highly Recommended
5    Business Analyst                    BankingPro             100.0    Highly Recommended


## Top 5 students for a job

In [12]:
job = jobs_df[jobs_df['job_id'] == 101].iloc[0]
top_studs = top_students_for_job(job, students_df, top_n=5)

print(f'Top 5 students for {job["title"]} @ {job["company"]}:\n')
print(f'{"#":<4} {"Name":<22} {"Score":<8} {"Skills":<8} {"Profile":<10} Status')
print('-'*75)
for i, r in enumerate(top_studs, 1):
    s = students_df[students_df['student_id'] == r.student_id].iloc[0]
    print(f'{i:<4} {s["name"]:<22} {r.match_score:<8} {r.skill_score:<8} {r.profile_score:<10} {r.status}')

Top 5 students for Data Scientist @ TechCorp India:

#    Name                   Score    Skills   Profile    Status
---------------------------------------------------------------------------
1    Aarav Shah             100.0    100.0    100.0      Highly Recommended
2    Rohan Mehta            100.0    100.0    100.0      Highly Recommended
3    Ananya Gupta           100.0    100.0    100.0      Highly Recommended
4    Meera Nair             100.0    100.0    100.0      Highly Recommended
5    Arjun Reddy            100.0    100.0    100.0      Highly Recommended


## Full match matrix

In [13]:
matrix = full_match_matrix(students_df, jobs_df)
print(f'matrix shape: {matrix.shape}  (students x jobs)')

flat = matrix.values.flatten()
print('\nscore distribution:')
buckets = [
    ('Not Recommended (<50)', 0, 50),
    ('Partial Match (50-69)', 50, 70),
    ('Recommended (70-84)', 70, 85),
    ('Highly Recommended (>=85)', 85, 101)
]
for label, lo, hi in buckets:
    n = int(np.sum((flat >= lo) & (flat < hi)))
    print(f'  {label:<35}: {n} pairs ({n/len(flat)*100:.1f}%)')

matrix shape: (30, 20)  (students x jobs)

score distribution:
  Not Recommended (<50)              : 206 pairs (34.3%)
  Partial Match (50-69)              : 33 pairs (5.5%)
  Recommended (70-84)                : 63 pairs (10.5%)
  Highly Recommended (>=85)          : 298 pairs (49.7%)


In [14]:
# peek at a slice
preview = matrix.iloc[:8, :5].copy()
preview.columns = [f'J{c.split("_")[1]}' for c in preview.columns]
preview

,J101,J102,J103,J104,J105
student_id,,,,,
1,100.00,100.00,88.57,100.00,100.0
2,31.11,63.08,28.57,95.00,30.0
3,100.00,100.00,100.00,100.00,100.0
4,26.66,41.92,0.00,27.86,60.0
5,91.11,87.70,73.57,77.14,80.0
6,100.00,100.00,88.57,100.00,100.0
7,8.89,32.30,0.00,37.86,30.0
8,100.00,100.00,73.57,100.00,100.0


## Evaluation metrics

In [15]:
metrics = evaluate_matching(students_df, jobs_df, threshold=70.0)

print('Matching Engine - Evaluation Results')
print(f'threshold = {metrics["threshold"]}, total pairs = {metrics["total_pairs"]}')
print()
print(f'Precision            : {metrics["precision"]}')
print(f'Recall               : {metrics["recall"]}')
print(f'False Positive Rate  : {metrics["false_positive_rate"]}')
print(f'F1 Score             : {metrics["f1_score"]}')
print(f'Coverage             : {metrics["coverage_pct"]}% of pairs recommended')
print()
print(f'Baseline precision   : {metrics["baseline_precision"]} (random matching)')
print(f'Our precision        : {metrics["precision"]}')
print(f'Improvement          : +{metrics["improvement_vs_baseline"]}')
print()
print(f'TP: {metrics["true_positives"]}  FP: {metrics["false_positives"]}  FN: {metrics["false_negatives"]}  TN: {metrics["true_negatives"]}')

Matching Engine - Evaluation Results
threshold = 70.0, total pairs = 600

Precision            : 0.6731
Recall               : 1.0
False Positive Rate  : 0.3305
F1 Score             : 0.8046
Coverage             : 60.17% of pairs recommended

Baseline precision   : 0.3 (random matching)
Our precision        : 0.6731
Improvement          : +0.3731

TP: 243  FP: 118  FN: 0  TN: 239


## Threshold sensitivity

In [16]:
# how do metrics change as we raise/lower the threshold
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<10} {"FPR":<10} {"F1":<10} Coverage')
print('-'*65)
for t in [50, 60, 70, 75, 80, 85, 90]:
    m = evaluate_matching(students_df, jobs_df, threshold=t)
    print(f'{t:<12} {m["precision"]:<12.4f} {m["recall"]:<10.4f} {m["false_positive_rate"]:<10.4f} {m["f1_score"]:<10.4f} {m["coverage_pct"]}%')

Threshold    Precision    Recall     FPR        F1         Coverage
-----------------------------------------------------------------


50           0.6168       1.0000     0.4230     0.7630     65.67%


60           0.6345       1.0000     0.3922     0.7764     63.83%
70           0.6731       1.0000     0.3305     0.8046     60.17%


75           0.6923       1.0000     0.3025     0.8182     58.5%


80           0.7297       1.0000     0.2521     0.8437     55.5%


85           0.8154       1.0000     0.1541     0.8983     49.67%
90           0.9240       1.0000     0.0560     0.9605     43.83%


## Edge cases

In [17]:
# weakest student vs most demanding job
weakest = students_df.iloc[students_df[SKILL_FEATURES].sum(axis=1).argmin()]
hardest = jobs_df.iloc[jobs_df[SKILL_FEATURES].sum(axis=1).argmax()]

r = calculate_match(weakest, hardest)
print(f'Student: {weakest["name"]} (lowest overall scores)')
print(f'Job    : {hardest["title"]} @ {hardest["company"]} (most requirements)')
print(f'Score  : {r.match_score}%  ->  {r.status}')
print(f'Missing: {r.missing_skills}')

Student: Pooja Bhatt (lowest overall scores)
Job    : ML Researcher @ RetailAI Corp (most requirements)
Score  : 8.0%  ->  Not Recommended
Missing: ['python', 'sql', 'machine_learning', 'data_visualization', 'cloud']


In [18]:
# best student vs easiest job
strongest = students_df.iloc[students_df[SKILL_FEATURES].sum(axis=1).argmax()]
easiest = jobs_df.iloc[jobs_df[SKILL_FEATURES].sum(axis=1).argmin()]

r2 = calculate_match(strongest, easiest)
print(f'Student: {strongest["name"]} (highest overall scores)')
print(f'Job    : {easiest["title"]} @ {easiest["company"]} (fewest requirements)')
print(f'Score  : {r2.match_score}%  ->  {r2.status}')

Student: Aditya Verma (highest overall scores)
Job    : Python Developer @ StartupXYZ (fewest requirements)
Score  : 100.0%  ->  Highly Recommended
